In [1]:
from torch.utils.data import Dataset, DataLoader
import numpy as np 
import torch 
import os 
data_path = os.path.join('data','UBFC-RPPG-Dataset')
subjects = os.listdir(data_path)  


sample_subject = subjects[0]
sample_roi = np.loadtxt(os.path.join(data_path, sample_subject, 'roi_colors.txt'), delimiter=',')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

In [2]:
#each frame has 3 labels in the dataset : Normalized Blood Volume Pulse , Heart Rate and Time. We will only train on the first one


class UBFC_Dataset(Dataset):
    def __init__(self, data_path, subjects,seq_len=10):
        self.data_path = data_path 
        self.subjects = subjects 
        self.seq_len = seq_len
        self.possible_ranges = [] 
        for subject in subjects:
            signal_path = os.path.join(data_path,subject, 'ground_truth.txt')
            signal = np.loadtxt(signal_path)
            num_starts = signal.shape[-1] - seq_len 
            for i in range(num_starts): 
                self.possible_ranges.append((subject, i)) 
    def __len__(self):
        return len(self.possible_ranges)

    def __getitem__(self, index):
        subject,i = self.possible_ranges[index]
        signal_path = os.path.join(self.data_path,subject, 'ground_truth.txt')
        colors_path = os.path.join(self.data_path, subject, 'roi_colors.txt')
        signals = np.loadtxt(signal_path)
        colors = np.loadtxt(colors_path, delimiter=',')
        signal_seq = signals[0,i : i + self.seq_len]
        color_seq = colors[i : i + self.seq_len]


        # --- MADE A BIG MISTAKE OF JUST NORMALIZING BY DIVIDING BY 255 AND THAT WAS CAUSING SATURATION
        
        min_vals = color_seq.min(axis=0, keepdims=True)
        max_vals = color_seq.max(axis=0, keepdims=True)
        color_seq = (color_seq - min_vals) / (max_vals - min_vals + 1e-6)
        # -------------------------------
        return torch.tensor(color_seq, dtype=torch.float32), torch.tensor(signal_seq, dtype=torch.float32)


In [3]:
batch_size = 256
seq_len = 128
dataset = UBFC_Dataset(data_path, subjects,seq_len=seq_len)

from  torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size 
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle=False)

sample_color_seq, sample_signal_seq = dataset[0]
print(sample_color_seq.shape) 

torch.Size([128, 9])


In [6]:
import torch.nn as nn

In [44]:
import torch
import torch.nn as nn

class FEN(nn.Module):
    def __init__(self, input_size=9, hidden_size=64, num_layers=3, output_size=1):
        super(FEN, self).__init__()
        #custom escrow rnn  
        self.core = nn.Sequential( 
            nn.Linear(hidden_size, hidden_size),  
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid()
        )
        self.x_proj = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size * 2, output_size)

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_size = output_size

    def forward(self, x):
        x = self.x_proj(x) 
        h = torch.zeros(x.size(0), x.size(1), self.hidden_size, device=x.device) 
        escrow = torch.zeros(x.size(0), x.size(1), self.hidden_size, device=x.device)

        for t in range(x.size(1)): 
            x_t = x[:, t, :] 
            if t == 0:
                h_prev = torch.zeros_like(x_t)
                escrow_prev = torch.zeros_like(x_t) 
            else:
                h_prev = h[:, t-1, :]      
                escrow_prev = escrow[:, t-1, :] 
                
            z = h_prev + x_t
     
            core_out = torch.tanh(self.core(z) + z) 
            gate_out = self.gate(core_out)   
            digested = gate_out * core_out
            
            
            h[:, t, :] = core_out - digested 
            escrow[:, t, :] = escrow_prev + digested 
            
        concatenated = torch.cat((h, escrow), dim=2) 
        reshaped = concatenated.view(-1, self.hidden_size * 2) 
        output = self.output(reshaped)
        
       
        return output.view(x.size(0), x.size(1))

In [47]:
model = FEN(input_size=9, hidden_size=64, num_layers=3, output_size=seq_len).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {n_params}")

Number of trainable parameters: 33792


In [ ]:
log_every = 10
num_epochs = 10 
val_every = 100

check_point_pth = os.path.join('checkpoints', 'best_fen.pth')
prev_best_val_loss = float('inf')
def save_checkpoint(model, optimizer, epoch, step, loss, path):
    checkpoint = {
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
        'epoch':epoch,
        'step':step,
        'loss':loss
    }
    torch.save(checkpoint, path)

model = FEN(input_size=9, hidden_size=64, num_layers=3, output_size=1)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = torch.nn.MSELoss() 
num_params = sum([param.numel() for param in model.parameters()])

print(f"Training  a  {num_params} parameters model on device: {device}")
def val():
    model.eval()
    total_loss = 0
    for i,(color_seq, signal_seq) in enumerate(test_loader):
        color_seq = color_seq.to(device)
        signal_seq = signal_seq.to(device)
        predictions = model(color_seq)
        loss = criterion(predictions.flatten(),signal_seq.flatten())
        total_loss += loss.item()
    model.train()
    return total_loss/len(test_loader)


for epoch in range(num_epochs):
    model.train()
    total_loss = 0  
    for i, (color_seq, signal_seq) in enumerate(train_loader):
        optimizer.zero_grad()
        color_seq = color_seq.to(device)
        signal_seq = signal_seq.to(device)
        predictions = model(color_seq) 
        
        loss = criterion(predictions.flatten(), signal_seq.flatten())
        
        loss = criterion(predictions.flatten(), signal_seq.flatten())
        loss.backward()
        optimizer.step() 
        if (i+1) % log_every == 0:
            
            if (i+1) % val_every ==0:
                val_loss = val()
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], train Loss: {loss.item():.4f}, val Loss: {val_loss:.4f}") 
                if val_loss < prev_best_val_loss:
                    print(f"New best model 💫💫 found.   loss: {val_loss:.4f}")  
                    save_checkpoint(model, optimizer, epoch, i, val_loss, check_point_pth)
                    prev_best_val_loss = val_loss
            else: 
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], train Loss: {loss.item():.4f}")

Training  a  17409 parameters model on device: cpu
